# 08 - Gold: Indicadores Mensuales (Empresas, Contratación y TRM)

Trabajo Práctico - Pipeline ETL end-to-end (Bronze → Silver / DQL → Gold)

| | Tabla |
|---|---|
| Entradas | `datos_empresas.gold.de_dim_empresas`, `datos_empresas.gold.de_fact_contratos`, `datos_empresas.silver.de_tasacambio` |
| Salida | `datos_empresas.gold.de_agg_indicadores_mes` |
| Grano | Una fila por mes (`anio_mes` = primer día del mes) |

**Depende de `06_Gold_Dim_Empresas`, `07_Gold_Fact_Contratos` y `05_Silver_TRM`.**

## Propósito analítico

Une en una sola serie de tiempo las tres fuentes que no comparten una llave de negocio, sino el **tiempo**. Responde:

* ¿Cuántas empresas se crean y se cancelan cada mes?
* ¿Influye la TRM en el cierre de empresas? Es una comparación entre la TRM promedio mensual y las cancelaciones.
* ¿Cómo evoluciona la contratación pública (cantidad, valor en COP y USD, contratación directa y pymes)?

Se construye con las tablas Gold de empresas y contratos, no directamente con Silver. Así los totales mensuales coinciden exactamente con la dimensión y el hecho, por ejemplo en la conversión a dólares.

> **Alcance de la TRM:** Bronze solo tiene la TRM de los últimos 12 meses, así que `trm_promedio` queda `NULL` en los meses anteriores. El análisis TRM vs. cancelaciones es exploratorio mientras no se cargue más historia.

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Medidas mensuales por fuente

In [ ]:
%python
dim = spark.table(GOLD_DIM_EMPRESAS)
fact = spark.table(GOLD_FACT_CONTRATOS)
trm = spark.table(SILVER_TRM)


def contar_si(condicion):
    return F.sum(F.when(condicion, 1).otherwise(0))


empresas_matriculadas = (dim.groupBy(F.trunc("fecha_matricula", "month").alias("anio_mes"))
                         .agg(F.count("*").alias("empresas_matriculadas")))

empresas_canceladas = (dim.filter(F.col("fecha_cancelacion").isNotNull())
                       .groupBy(F.trunc("fecha_cancelacion", "month").alias("anio_mes"))
                       .agg(F.count("*").alias("empresas_canceladas")))

contratacion = (fact.groupBy(F.col("anio_mes_firma").alias("anio_mes"))
                .agg(F.count("*").alias("contratos_firmados"),
                     F.sum("valor_contrato").alias("valor_contratado_cop"),
                     F.sum("valor_contrato_usd").alias("valor_contratado_usd"),
                     contar_si(F.col("es_contratacion_directa")).alias("contratos_directos"),
                     contar_si(F.col("es_pyme")).alias("contratos_pyme")))

tasa_cambio = (trm.groupBy(F.trunc("fecha_vigencia_desde", "month").alias("anio_mes"))
               .agg(F.round(F.avg("valor_trm"), 2).cast("decimal(12,2)").alias("trm_promedio"),
                    F.min("valor_trm").alias("trm_minima"),
                    F.max("valor_trm").alias("trm_maxima")))

---
## Paso 2: Unir por mes y calcular indicadores derivados

`FULL OUTER JOIN` por `anio_mes` para no perder ningún mes que tenga datos en al menos una fuente. Los conteos ausentes se convierten en 0; la TRM ausente queda `NULL`, porque 0 no tendría sentido.

In [ ]:
%python
df_mes = (empresas_matriculadas
          .join(empresas_canceladas, "anio_mes", "full")
          .join(contratacion, "anio_mes", "full")
          .join(tasa_cambio, "anio_mes", "full")
          .filter(F.col("anio_mes").isNotNull()))

for columna in ["empresas_matriculadas", "empresas_canceladas", "contratos_firmados", "contratos_directos", "contratos_pyme"]:
    df_mes = df_mes.withColumn(columna, F.coalesce(F.col(columna), F.lit(0)).cast("bigint"))

w_tiempo = Window.orderBy("anio_mes")

df_agg = (df_mes
          .withColumn("anio", F.year("anio_mes"))
          .withColumn("mes", F.month("anio_mes"))
          .withColumn("saldo_neto_empresas", F.col("empresas_matriculadas") - F.col("empresas_canceladas"))
          .withColumn("pct_contratacion_directa",
                      F.when(F.col("contratos_firmados") > 0,
                             F.round(100 * F.col("contratos_directos") / F.col("contratos_firmados"), 2)))
          .withColumn("variacion_trm_pct",
                      F.round(100 * (F.col("trm_promedio") / F.lag("trm_promedio").over(w_tiempo) - 1), 2))
          .select("anio_mes", "anio", "mes",
                  "empresas_matriculadas", "empresas_canceladas", "saldo_neto_empresas",
                  "contratos_firmados", "contratos_directos", "pct_contratacion_directa", "contratos_pyme",
                  "valor_contratado_cop", "valor_contratado_usd",
                  "trm_promedio", "trm_minima", "trm_maxima", "variacion_trm_pct"))

df_agg = df_agg.withColumn("_hash_registro", hash_columnas([c for c in df_agg.columns]))

---
## Paso 3: Escritura idempotente (MERGE por `anio_mes`)

In [ ]:
%python
m_gold = escribir_merge(
    df_agg, GOLD_AGG_MES, ["anio_mes"], capa="gold",
    comentario="Gold - Indicadores mensuales: creacion y cancelacion de empresas, contratacion publica y TRM promedio.",
)

# Control de consistencia: los totales del agregado deben cuadrar con la dimensión y el hecho
totales = spark.table(GOLD_AGG_MES).agg(F.sum("empresas_matriculadas").alias("m"), F.sum("contratos_firmados").alias("c")).collect()[0]
assert totales["m"] == dim.filter(F.col("fecha_matricula").isNotNull()).count(), "Las matrículas mensuales no cuadran con la dimensión"
assert totales["c"] == fact.filter(F.col("fecha_firma").isNotNull()).count(), "Los contratos mensuales no cuadran con el hecho"
print(f"Totales conciliados: {totales['m']:,} matrículas y {totales['c']:,} contratos")

---
## Paso 4: Consultas de ejemplo

In [ ]:
%sql
-- TRM vs. creación y cierre de empresas (meses con TRM disponible)
SELECT anio_mes, trm_promedio, variacion_trm_pct,
       empresas_matriculadas, empresas_canceladas, saldo_neto_empresas
FROM datos_empresas.gold.de_agg_indicadores_mes
WHERE trm_promedio IS NOT NULL
ORDER BY anio_mes;

In [ ]:
%sql
-- Correlación exploratoria (Pearson) entre la TRM y las cancelaciones / matrículas.
-- Se calcula como covarianza / (desv_x * desv_y) con try_divide: con ANSI activado, CORR falla
-- si alguna serie no varía; así devuelve NULL en ese caso.
SELECT COUNT(*) AS meses,
       ROUND(TRY_DIVIDE(COVAR_POP(trm_promedio, empresas_canceladas),
                        STDDEV_POP(trm_promedio) * STDDEV_POP(empresas_canceladas)), 3) AS corr_trm_cancelaciones,
       ROUND(TRY_DIVIDE(COVAR_POP(trm_promedio, empresas_matriculadas),
                        STDDEV_POP(trm_promedio) * STDDEV_POP(empresas_matriculadas)), 3) AS corr_trm_matriculas
FROM datos_empresas.gold.de_agg_indicadores_mes
WHERE trm_promedio IS NOT NULL;

In [ ]:
%python
finalizar({"notebook": "08_Gold_Agg_Indicadores_Mes", "gold": m_gold})